# Sifu demo: Design stage, risk router, pause and resume

Needs Ollama running with the models in `config.yaml` and `tripplanner-ios` at `workspace.product_repo_path`. Cell 2 runs the five seeded Design-stage tickets. Cells 3 to 5 show the router: a low-risk ticket is approved automatically and a ticket that names a protected file (`.github/`) pauses for you.

The databases in `data/` persist between runs, so tickets and open requests pile up. For a clean run, set `RESET = True` in the next cell, restart the kernel and run it before the others.

In [ ]:
from pathlib import Path

RESET = False  # True deletes the demo databases and their checkpoints for a clean run

if RESET:
    # Restart the kernel first so no open connection still holds these files.
    for name in ("design.db", "sifu.db"):
        for path in Path("data").glob(f"{name}*"):  # the .db file plus its -wal and -shm
            path.unlink()
            print(f"removed {path}")

In [ ]:
from sifu.demo import run_design_demo

rows = run_design_demo(db_path="data/design.db")
rows

In [ ]:
from sifu.demo import open_demo, run_router_demo

deps, project_id = open_demo(db_path="data/sifu.db")
ids = run_router_demo(deps, project_id)

The second ticket is paused in the SQLite checkpoint. Look at the open request. If it asks you to approve, answer `approve` or `close`, or any other text as feedback. If it asks you to answer questions, answer with text (it goes back to the reviewers as a clarification) or `close`.

In [ ]:
from sifu import db
from sifu.demo import answer_request

# Answer the request this run's protected ticket opened. A reused database file can hold
# older open requests too, so pick it by ticket rather than by a fixed id.
[request] = [r for r in db.open_requests(deps.conn) if r["ticket_id"] == ids["protected_path"]]
state = answer_request(deps, request["id"], "approve")